# 02b. Fine-Tuning Incremental: Especialização em Placas Brasileiras

Este notebook realiza o **Treinamento Incremental** do detector YOLOv8 a partir do modelo treinado no notebook `02`, especializando-o no dataset de **placas veiculares brasileiras** (Jhonathas Moura / IFRN).

### Por que este notebook existe?
- O modelo base aprendeu a geometria geral de 8.800 placas no notebook `02`.
- Aqui, usamos **taxa de aprendizado controlada (`lr0=0.001`)** para refinar a atenção da rede nas características de veículos e placas brasileiras (padronização Mercosul e cinza antiga, iluminações reais do país) sem sofrer *esquecimento catastrófico*.
- Atende ao experimento **EXP-07** do planejamento do projeto.

--- 
## 1. Instalação e Checagem da GPU
*Certifique-se de que a GPU está ativa em: `Ambiente de Execução` -> `Alterar tipo de ambiente de execução` -> `T4 GPU`.*

In [ ]:
!pip install -q ultralytics onnx onnxruntime

import torch
print('Versão do PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('✅ GPU detectada:', torch.cuda.get_device_name(0))
else:
    print('⚠️ ATENÇÃO: Nenhuma GPU detectada. Ative a GPU no menu Ambiente de Execução!')

--- 
## 2. Carregar o Checkpoint Anterior (`best.pt`)
Se você estiver executando no mesmo ambiente do notebook `02`, ele detecta o modelo automaticamente. Caso contrário, permite fazer o upload do `best.pt` do seu computador.

In [ ]:
from pathlib import Path
from google.colab import files

# Tenta localizar o best.pt da sessão atual
candidatos = list(Path('/content').glob('**/treino_placas/weights/best.pt'))
if not candidatos:
    candidatos = list(Path('/content').glob('**/best.pt'))

if candidatos:
    caminho_base = str(candidatos[0])
    print('✅ Checkpoint do treino anterior encontrado:', caminho_base)
else:
    print('Checkpoint anterior não encontrado no disco local do Colab.')
    print('Por favor, faça o upload do arquivo best.pt gerado no primeiro treino:')
    upload = files.upload()
    caminho_base = list(upload.keys())[0]
    print('✅ Checkpoint carregado com sucesso:', caminho_base)

--- 
## 3. Download e Organização do Dataset Brasileiro (400 Imagens)
Baixa o arquivo compactado diretamente do repositório público do IFRN e organiza a estrutura de diretórios para o padrão YOLOv8.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

print('Baixando dataset.zip do GitHub (jhonathasmoura/Deteccao_Placas_Veiculos_YOLOv4)...')
url = 'https://raw.githubusercontent.com/jhonathasmoura/Deteccao_Placas_Veiculos_YOLOv4/main/dataset.zip'
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})

with urllib.request.urlopen(req) as resp:
    zip_bytes = resp.read()

print(f'Download concluído: {len(zip_bytes) / (1024 * 1024):.2f} MB')

# Extrai arquivos brutos
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    z.extractall('/content/dataset_brasil_raw')

# Estruturação nas pastas YOLOv8
base_dest = Path('/content/dataset_brasil')
mapeamento = [
    ('train', 'train'),
    ('valid', 'val'),
    ('test', 'test')
]

for pasta_origem, split_dest in mapeamento:
    img_dir = base_dest / 'images' / split_dest
    lbl_dir = base_dest / 'labels' / split_dest
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    
    origem = Path(f'/content/dataset_brasil_raw/{pasta_origem}')
    for arq in origem.glob('*.*'):
        if arq.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            arq.replace(img_dir / arq.name)
        elif arq.suffix.lower() == '.txt':
            arq.replace(lbl_dir / arq.name)

# Configuração placas_brasil.yaml
yaml_brasil = '''
path: /content/dataset_brasil
train: images/train
val: images/val
test: images/test

names:
  0: placa
'''

with open('/content/placas_brasil.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_brasil.strip())

print('✅ Dataset brasileiro estruturado com sucesso!')

---
## 3.1 Aumento de Dados Offline (Opcional: Duplicação do Dataset Brasileiro)
Como o dataset brasileiro possui cerca de 400 imagens, duplicá-lo ou triplicá-lo offline gera mais exemplos com variações de iluminação, ruído noturno e borrão de movimento sem risco de desalinhar as anotações.

In [ ]:
import cv2
import random
import shutil
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# =====================================================================
# CONFIGURAÇÃO DE DUPLICAÇÃO FLEXÍVEL
# =====================================================================
DUPLICAR_DATASET = True   # Alterne para True para duplicar ou False para pular
MULTIPLICADOR = 2         # 2 = duplica (1 cópia nova por imagem), 3 = triplica
# =====================================================================

def aplicar_variacao_realista(img: np.ndarray, tipo: int) -> np.ndarray:
    """Gera variações fotométricas típicas de cenários reais de ALPR."""
    im = img.copy()
    if tipo == 0:
        # Cenário 1: Noite / Baixa iluminação com ruído sutil
        gamma = random.uniform(0.50, 0.70)
        inv_gamma = 1.0 / gamma
        tabela = np.array([((i / 255.0) ** inv_gamma) * 255 for i in range(256)]).astype('uint8')
        im = cv2.LUT(im, tabela)
        ruido = np.random.normal(0, 10, im.shape).astype(np.float32)
        im = np.clip(im.astype(np.float32) + ruido, 0, 255).astype(np.uint8)
    elif tipo == 1:
        # Cenário 2: Movimento rápido do veículo (Motion Blur horizontal)
        k = random.choice([5, 7])
        kernel = np.zeros((k, k))
        kernel[int((k - 1) / 2), :] = np.ones(k) / k
        im = cv2.filter2D(im, -1, kernel)
    elif tipo == 2:
        # Cenário 3: Sol intenso / Contraste e brilho elevados
        im = cv2.convertScaleAbs(im, alpha=random.uniform(1.2, 1.4), beta=random.uniform(10, 25))
    else:
        # Cenário 4: Granulação de sensor de segurança (ISO alto)
        ruido = np.random.normal(0, 16, im.shape).astype(np.float32)
        im = np.clip(im.astype(np.float32) + ruido, 0, 255).astype(np.uint8)
    return im

if DUPLICAR_DATASET:
    train_img_dir = Path('/content/dataset_brasil/images/train')
    train_lbl_dir = Path('/content/dataset_brasil/labels/train')
    
    originais = [f for f in train_img_dir.glob('*.*') if f.suffix.lower() in ['.jpg', '.png', '.jpeg'] and not f.stem.endswith(('_aug1', '_aug2'))]
    print(f'⚙️ Gerando variações realistas para {len(originais)} imagens de treino brasileiras...')
    
    for arq_img in tqdm(originais):
        im = cv2.imread(str(arq_img))
        if im is None:
            continue
        arq_lbl = train_lbl_dir / f'{arq_img.stem}.txt'
        tem_lbl = arq_lbl.exists()
        
        for k in range(1, MULTIPLICADOR):
            tipo = (hash(arq_img.stem) + k) % 4
            im_aug = aplicar_variacao_realista(im, tipo)
            cv2.imwrite(str(train_img_dir / f'{arq_img.stem}_aug{k}.jpg'), im_aug)
            if tem_lbl:
                shutil.copyfile(arq_lbl, train_lbl_dir / f'{arq_img.stem}_aug{k}.txt')
                
    total = len([f for f in train_img_dir.glob('*.*') if f.suffix.lower() in ['.jpg', '.png', '.jpeg']])
    print(f'✅ Dataset brasileiro ampliado com sucesso! Total agora: {total} imagens.')
else:
    print('ℹ️ Duplicação offline desativada. Treinando com o conjunto original.')

--- 
## 4. Fine-Tuning Incremental
Iniciamos a partir do `best.pt` anterior com `lr0=0.001` (taxa suave para ajuste fino).

In [ ]:
from ultralytics import YOLO

modelo_incremental = YOLO(caminho_base)

# Fine-tuning com aumento dinâmico completo na GPU
resultados = modelo_incremental.train(
    data='/content/placas_brasil.yaml',
    epochs=25,
    imgsz=640,
    batch=16,
    # --- Data Augmentation Dinâmico: ---
    degrees=25.0,       # Rotação aleatória de +/- 25° para placas diagonais
    shear=10.0,         # Cisalhamento angular (câmeras de lado/postes)
    perspective=0.001,  # Distorção de perspectiva 3D realista
    hsv_v=0.4,          # Variação forte de brilho (noite/sombra/sol)
    hsv_s=0.5,          # Variação de saturação de cor
    translate=0.1,      # Deslocamento no quadro
    scale=0.5,          # Escala (zoom in / zoom out)
    mosaic=1.0,         # Composição de 4 imagens em 1
    # -----------------------------------
    lr0=0.001,
    patience=10,
    project='runs/detect',
    name='treino_brasil',
    verbose=True
)

--- 
## 5. Avaliação das Métricas em Placas Brasileiras

In [ ]:
from pathlib import Path
from ultralytics import YOLO

candidato_final = list(Path('/content').glob('**/treino_brasil*/weights/best.pt'))[0]
modelo_final = YOLO(str(candidato_final))

metricas = modelo_final.val(data='/content/placas_brasil.yaml', split='test')

print('=' * 55)
print('MÉTRICAS DO MODELO ESPECIALIZADO NO BRASIL')
print('=' * 55)
print(f'mAP@0.5:      {metricas.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metricas.box.map:.4f}')
print(f'Precisão:     {metricas.box.mp:.4f}')
print(f'Recall:       {metricas.box.mr:.4f}')
print('=' * 55)

--- 
## 6. Exportação para Formato ONNX

In [ ]:
caminho_onnx_brasil = modelo_final.export(
    format='onnx',
    imgsz=640,
    opset=12,
    simplify=True
)
print('✅ Modelo ONNX especializado exportado em:', caminho_onnx_brasil)

--- 
## 7. Verificação com onnxruntime em CPU

In [ ]:
import onnxruntime as ort

sessao = ort.InferenceSession(caminho_onnx_brasil, providers=['CPUExecutionProvider'])
entrada = sessao.get_inputs()[0]
saida = sessao.get_outputs()[0]

print('Entrada ONNX:', entrada.name, entrada.shape)
print('Saída ONNX:  ', saida.name, saida.shape)
print('✅ Modelo verificado com sucesso!')

--- 
## 8. Download do Modelo Final para a Aplicação Desktop

In [ ]:
from google.colab import files

print(f'Iniciando download de: {caminho_onnx_brasil}')
files.download(caminho_onnx_brasil)